# 🌥️ Kinnu Cartoon Backgrounds — free cloud GPU (Kaggle / Colab)

Generate **1920×1080 no-character scene backgrounds** with **SDXL** on a free GPU, then
download `kinnu_backgrounds.zip` and unzip into `assets/cartoon/backgrounds/` in the
Shiksha-Cast repo. The cutout engine animates the consistent characters on top.

**Kaggle:** Settings → Accelerator → **GPU T4 ×1** (free ~30 hrs/week). **Colab:** Runtime → GPU.

Edit the `PROMPTS` list, run all cells, download the zip. ~10–20s per image on T4.

In [ ]:
# 1) Install (torch is preinstalled on Kaggle/Colab GPU runtimes)
!pip install -q --upgrade diffusers transformers accelerate safetensors

In [ ]:
# 2) Load SDXL (fp16, fits a free 16 GB T4/P100)
import torch
from diffusers import StableDiffusionXLPipeline

pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16, use_safetensors=True, variant='fp16',
).to('cuda')
pipe.enable_attention_slicing()
pipe.set_progress_bar_config(disable=True)
print('SDXL ready on', torch.cuda.get_device_name(0))

In [ ]:
# 3) Style lock + your scene prompts (NO characters; leave space for the cast)
STYLE = ('flat 2D cartoon background, preschool kids cartoon style, bright pastel colors, '
         'thick clean outlines, simple shapes, soft lighting, cheerful, empty foreground, '
         'wide shot, no characters, no people, no text')
NEG = ('people, person, child, kid, character, mascot, face, hands, text, words, watermark, '
       'logo, signature, blurry, low quality, deformed, cluttered, dark, scary')

# One entry per background you want. Rename/extend freely.
PROMPTS = [
    'a bright cheerful preschool classroom with a green chalkboard, colorful shelves and a window',
    'a sunny playground with swings, a slide, green grass and a blue sky with fluffy clouds',
    'a cozy kitchen with a sink, soap and a window, clean and bright',
    'a magical night sky with a glowing golden star bridge arcing across, twinkling stars',
    'a colorful park with flowers, trees, a path and a pond on a sunny day',
    'a tidy kids bedroom with a bed, toys in a basket and a bookshelf',
]

In [ ]:
# 4) Generate -> 1920x1080 PNGs
import os
os.makedirs('backgrounds', exist_ok=True)
GEN_W, GEN_H = 1344, 768   # SDXL-friendly 16:9, upscaled to 1920x1080 on save
for i, p in enumerate(PROMPTS, 1):
    img = pipe(prompt=f'{p}, {STYLE}', negative_prompt=NEG,
               width=GEN_W, height=GEN_H, num_inference_steps=30, guidance_scale=6.5).images[0]
    img = img.resize((1920, 1080))
    out = f'backgrounds/bg_{i:03d}.png'
    img.save(out)
    print('saved', out)

In [ ]:
# 5) Zip for download
import shutil
shutil.make_archive('kinnu_backgrounds', 'zip', 'backgrounds')
print('Done -> kinnu_backgrounds.zip')
# Kaggle: find it in the right-hand Output panel. Colab: run the next line.
try:
    from google.colab import files; files.download('kinnu_backgrounds.zip')
except Exception:
    pass

## Use the backgrounds
1. Download `kinnu_backgrounds.zip` and unzip the PNGs into `assets/cartoon/backgrounds/`.
2. In a `scenes.yaml`, set `background: bg_001.png` (etc.) per scene.
3. Run locally: `python -m shiksha_cast cartoon-build -c <episode>` — the cast animates on top.

Tip: keep characters in the lower-left/right, so prompts that leave the lower area open work best.
For a cinematic *intro* (no characters), you can swap SDXL for a Wan 2.2 video cell later.